In [1]:
import sys
import os
for p in list(sys.path):
    if 'recurrent-memory-transformer' in p:
        sys.path.remove(p)

for e in sys.path:
    print(e)

/home/griver/projects/ml/nlp/multi-step-retrieval-rl/notebooks
/home/griver/anaconda3/envs/rmt/lib/python39.zip
/home/griver/anaconda3/envs/rmt/lib/python3.9
/home/griver/anaconda3/envs/rmt/lib/python3.9/lib-dynload

/home/griver/.local/lib/python3.9/site-packages
/home/griver/anaconda3/envs/rmt/lib/python3.9/site-packages


In [1]:
import sys
import os

%load_ext autoreload
%autoreload 2

repo_dir = os.path.dirname(os.path.abspath("./"))
if repo_dir not in sys.path:
    print(f'add repository dir: {repo_dir}')
    sys.path.append(repo_dir)

from rl import WordsCounterEnv, ReplayAdapter
from rl.bert_predictor import BertPredictor
from datasets import load_dataset
from datasets.dataset_dict import DatasetDict, Dataset
from transformers import RobertaTokenizer, RobertaModel, AutoModel, AutoTokenizer, PreTrainedTokenizer, PreTrainedTokenizerFast
import numpy as np
from torch import nn, Tensor
import torch
from copy import deepcopy
from typing import Dict
import os

os.environ['TOKENIZERS_PARALLELISM'] = 'true'
print("loading dataset: AIRI-NLP/quality_counter_new_1024")
dataset = load_dataset("AIRI-NLP/quality_counter_new_1024")

add repository dir: /home/griver/projects/ml/nlp/multi-step-retrieval-rl


/home/griver/anaconda3/envs/rmt/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


loading dataset: AIRI-NLP/quality_counter_new_1024


In [2]:
bert_name = "haisongzhang/roberta-tiny-cased"
tokenizer = AutoTokenizer.from_pretrained(bert_name, use_fast=True, revision="main")
bert_model = AutoModel.from_pretrained(bert_name, revision="main")
predictor = BertPredictor(bert_model, 4, tokenizer, 512, 256, 1).cuda()

env = WordsCounterEnv(dataset, block_size=32, block_embedding=predictor, max_length=1024, tokenizer=tokenizer)

In [3]:
buffer = ReplayAdapter(1000_000, tokenizer = tokenizer)

In [5]:
from tqdm import tqdm

for _ in tqdm(range(100_000)):

    s = env.reset()

    for i in range(10):
        s_next, memory_block, reward, done = env.step(i)
        buffer.add(s, memory_block, s_next, reward, done)
        s = s_next

100%|███████████████████████████████████| 100000/100000 [35:16<00:00, 47.24it/s]


In [6]:
s_memory, a, next_s_memory, embeds, r, not_done = buffer.sample(4)

In [7]:
s_memory

Memory(block_ids=[[0, 1, 2, 3, 4, 5, 6, 7], [0, 1, 2, 3, 4, 5, 6, 7], [0, 1, 2, 3, 4, 5], []], input_ids=tensor([[1293, 1242, 1551,  ..., 1902, 1852, 4550],
        [1293, 1242, 1551,  ..., 1231, 1177, 4044],
        [1293, 1242, 1551,  ...,    0,    0,    0],
        [ 102,    0,    0,  ...,    0,    0,    0]]), attention_mask=tensor([[1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 0, 0,  ..., 0, 0, 0]]), text=['how many times the word\'cigar\'appears in the text? [SEP] though. " " that\'s good. give her the run of the deck, and [SEP] take care of her. " " yes, sir, we will, " answered sampson, as respectfully as though it were a legitimate order - - [SEP] for force of habit is strong. then he left the room, locking the door behind him. denman smoked until he had finished the cigar, and, after [SEP] he had eaten a supper brought by billings, he smoked again until darkness closed down. and with the closing down of darkness came 

In [8]:
a

MemoryBlock(index=[8, 8, 6, 0], input_ids=tensor([[ 5018,   117,  4040,  1120,  1675,  1852,  1111,  5821, 10681,   117,
          1177,  1119,  8703,  1103,  1981,  6077,   119,  1112,  1111,  5821,
         10681,  1105,  5358,  5213,  3447,  1125,  1276,   117,  1175,  1108,
          1720,  1175],
        [  119,  1128,   112,  1231,  3271,  1104,  5993,  1183,  4044,   117,
          1176,   178,  1821,   119,  1175,   112,   188,  1160, 12072,   117,
          1128,  1221,   119,  1141,   112,   188,  1176, 14458,   131,  1119,
           112,   188],
        [ 1437,  1107,  1199,  3049,  2496,  1184,  1142,  4098,  3362,  2086,
           119,   107,  8792,  1977,   178,   119,  1139, 13387, 25550,   119,
          7806, 25550,  1182,   119,   179, 10559, 15703,   112,   188,  1981,
           178,  1964],
        [ 1293,  1242,  1551,  1103,  1937,   112,   183,   112,   189,   112,
          2691,  1107,  1103,  3087,   136,   102,  1106,  1210,  1317,  6961,
          1279,  

In [9]:
embeds.shape, r.shape, not_done.shape 

(torch.Size([4, 32, 256]), torch.Size([4, 1]), torch.Size([4, 1]))